In [49]:
pip install pyarrow fastparquet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [50]:
import os
import pandas as pd

EXPORT_DIR = r"D:\256_group_dataset\phase1_outputs"

TRAIN_PATH = os.path.join(EXPORT_DIR, "interactions_model_ready_train.parquet")
VALID_PATH = os.path.join(EXPORT_DIR, "interactions_model_ready_valid.parquet")
TEST_PATH  = os.path.join(EXPORT_DIR, "interactions_model_ready_test.parquet")

print("Train path:", TRAIN_PATH)
print("Valid path:", VALID_PATH)
print("Test path: ", TEST_PATH)

Train path: D:\256_group_dataset\phase1_outputs\interactions_model_ready_train.parquet
Valid path: D:\256_group_dataset\phase1_outputs\interactions_model_ready_valid.parquet
Test path:  D:\256_group_dataset\phase1_outputs\interactions_model_ready_test.parquet


In [51]:
train_df = pd.read_parquet(TRAIN_PATH, engine="fastparquet")
valid_df = pd.read_parquet(VALID_PATH, engine="fastparquet")
test_df  = pd.read_parquet(TEST_PATH,  engine="fastparquet")

In [52]:
# Normalize schema so the rest of the notebook can use consistent names

def normalize_interaction_schema(df):
    df = df.copy()

    # If exported files use original_user_id / original_item_id, rename them
    if "original_user_id" in df.columns and "user_id" not in df.columns:
        df["user_id"] = df["original_user_id"]

    if "original_item_id" in df.columns and "item_id" not in df.columns:
        df["item_id"] = df["original_item_id"]

    return df

train_df = normalize_interaction_schema(train_df)
valid_df = normalize_interaction_schema(valid_df)
test_df  = normalize_interaction_schema(test_df)

In [53]:
# Quick schema / preview check

print("TRAIN COLUMNS:", list(train_df.columns))
print("VALID COLUMNS:", list(valid_df.columns))
print("TEST COLUMNS: ", list(test_df.columns))

display(train_df.head())

TRAIN COLUMNS: ['interaction_id', 'user_idx', 'item_idx', 'rating', 'timestamp', 'split', 'original_user_id', 'original_item_id', 'user_id', 'item_id']
VALID COLUMNS: ['interaction_id', 'user_idx', 'item_idx', 'rating', 'timestamp', 'split', 'original_user_id', 'original_item_id', 'user_id', 'item_id']
TEST COLUMNS:  ['interaction_id', 'user_idx', 'item_idx', 'rating', 'timestamp', 'split', 'original_user_id', 'original_item_id', 'user_id', 'item_id']


,interaction_id,user_idx,item_idx,rating,timestamp,split,original_user_id,original_item_id,user_id,item_id
0,37347498,711617,206723,1.0,2002-07-01,train,Lorraine7,Hotel_Review-g34172-d88311-Reviews-Best_Wester...,Lorraine7,Hotel_Review-g34172-d88311-Reviews-Best_Wester...
1,13529409,1649570,269376,1.0,2002-08-01,train,mzani,Hotel_Review-g60750-d80133-Reviews-Handlery_Ho...,mzani,Hotel_Review-g60750-d80133-Reviews-Handlery_Ho...
2,19927246,713942,208141,5.0,2002-08-01,train,Louisville40204,Hotel_Review-g34439-d145476-Reviews-Sherbrooke...,Louisville40204,Hotel_Review-g34439-d145476-Reviews-Sherbrooke...
3,28220531,343920,30997,4.0,2002-08-01,train,Cerce,Hotel_Review-g153292-d153995-Reviews-Hotel_Bar...,Cerce,Hotel_Review-g153292-d153995-Reviews-Hotel_Bar...
4,1528489,1607657,116123,4.0,2002-09-01,train,mcguillicudy,Hotel_Review-g264369-d148789-Reviews-Lazy_Parr...,mcguillicudy,Hotel_Review-g264369-d148789-Reviews-Lazy_Parr...


In [54]:
# Basic sanity checks to make sure the popularity notebook is using the expected pipeline outputs

required_cols = ["interaction_id", "user_id", "item_id", "rating", "timestamp", "split"]

for name, df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    missing = [c for c in required_cols if c not in df.columns]
    print(f"{name} missing columns:", missing)

    print(f"\n{name.upper()} SUMMARY")
    print("rows:", len(df))
    print("unique users:", df["user_id"].nunique())
    print("unique items:", df["item_id"].nunique())
    print("rating min/max:", df["rating"].min(), df["rating"].max())
    print("split values:", df["split"].unique())
    print("timestamp min/max:", df["timestamp"].min(), df["timestamp"].max())
    print("-" * 50)

train missing columns: []

TRAIN SUMMARY
rows: 16064988
unique users: 1927834
unique items: 309395
rating min/max: 1.0 5.0
split values: ['train']
timestamp min/max: 2002-07-01 00:00:00 2019-05-08 00:00:00
--------------------------------------------------
valid missing columns: []

VALID SUMMARY
rows: 1927834
unique users: 1927834
unique items: 253644
rating min/max: 1.0 5.0
split values: ['valid']
timestamp min/max: 2002-10-01 00:00:00 2019-05-11 00:00:00
--------------------------------------------------
test missing columns: []

TEST SUMMARY
rows: 1927834
unique users: 1927834
unique items: 247137
rating min/max: 1.0 5.0
split values: ['test']
timestamp min/max: 2002-10-01 00:00:00 2019-09-20 00:00:00
--------------------------------------------------


In [55]:
# SECTION 2 — Compute train-only item popularity statistics

# We will build the popularity model using TRAIN data only.
# This avoids leakage from validation/test.

train_item_stats = (
    train_df
    .groupby("item_id", as_index=False)
    .agg(
        item_review_count=("rating", "size"),
        item_mean_rating=("rating", "mean")
    )
)

global_mean_rating = train_df["rating"].mean()

print("Global train mean rating:", round(global_mean_rating, 6))
print("Number of items in train stats:", len(train_item_stats))

display(train_item_stats.head())

Global train mean rating: 4.071144
Number of items in train stats: 309395


,item_id,item_review_count,item_mean_rating
0,Hotel_Review-g10006284-d1083311-Reviews-The_Re...,163,4.693252
1,Hotel_Review-g10006284-d151184-Reviews-Club_Me...,734,3.892371
2,Hotel_Review-g10006284-d151225-Reviews-Ports_o...,285,4.035088
3,Hotel_Review-g10006284-d151307-Reviews-The_San...,523,4.292543
4,Hotel_Review-g10006284-d185069-Reviews-Point_G...,106,4.528302


In [56]:
# Choose the smoothing constant m
# Recommendation: median item review count in TRAIN
# Reason:
# - simple and data-driven
# - shrinks low-support items toward the global mean
# - keeps this baseline stable without making it complicated

m = int(train_item_stats["item_review_count"].median())

print("Smoothing constant m (median item review count):", m)

Smoothing constant m (median item review count): 18


In [57]:
# Compute the Bayesian-smoothed popularity score
#
# score_i = (v / (v + m)) * R + (m / (v + m)) * C
#
# where:
# v = item review count in train
# R = item mean rating in train
# C = global train mean rating

train_item_stats["popularity_score"] = (
    (train_item_stats["item_review_count"] / (train_item_stats["item_review_count"] + m)) * train_item_stats["item_mean_rating"]
    + (m / (train_item_stats["item_review_count"] + m)) * global_mean_rating
)

display(train_item_stats.head())

,item_id,item_review_count,item_mean_rating,popularity_score
0,Hotel_Review-g10006284-d1083311-Reviews-The_Re...,163,4.693252,4.631385
1,Hotel_Review-g10006284-d151184-Reviews-Club_Me...,734,3.892371,3.896650
2,Hotel_Review-g10006284-d151225-Reviews-Ports_o...,285,4.035088,4.037230
3,Hotel_Review-g10006284-d151307-Reviews-The_San...,523,4.292543,4.285177
4,Hotel_Review-g10006284-d185069-Reviews-Point_G...,106,4.528302,4.461940


In [58]:
# Create the global popularity ranking
# Highest score first, then more reviews, then higher mean rating, then item_id for stable ordering

popularity_ranking_df = (
    train_item_stats
    .sort_values(
        by=["popularity_score", "item_review_count", "item_mean_rating", "item_id"],
        ascending=[False, False, False, True]
    )
    .reset_index(drop=True)
)

# Add explicit rank column
popularity_ranking_df["pop_rank"] = range(1, len(popularity_ranking_df) + 1)

display(popularity_ranking_df.head(20))

,item_id,item_review_count,item_mean_rating,popularity_score,pop_rank
0,Hotel_Review-g274707-d658737-Reviews-Hotel_Res...,815,4.968098,4.948716,1
1,Hotel_Review-g188671-d1534996-Reviews-Huis_Kon...,215,5.000000,4.928243,2
2,Hotel_Review-g274887-d635259-Reviews-Kapital_I...,287,4.979094,4.925510,3
3,Hotel_Review-g188671-d590867-Reviews-Cote_Cana...,297,4.976431,4.924700,4
4,Hotel_Review-g298564-d1547516-Reviews-Hotel_Mu...,334,4.964072,4.918411,5
5,Hotel_Review-g186612-d213125-Reviews-Friars_Gl...,458,4.949782,4.916556,6
6,Hotel_Review-g293924-d7180030-Reviews-Hanoi_La...,974,4.929158,4.913589,7
7,Hotel_Review-g186370-d2001897-Reviews-Hawkins_...,196,4.989796,4.912526,8
8,Hotel_Review-g471846-d646675-Reviews-Dulini_Lo...,214,4.981308,4.910692,9
9,Hotel_Review-g188675-d2173238-Reviews-Main_Str...,211,4.981043,4.909522,10


In [59]:
# Quick sanity checks

print("Popularity ranking size:", len(popularity_ranking_df))
print("Any missing scores?:", popularity_ranking_df["popularity_score"].isna().sum())
print("Any missing item IDs?:", popularity_ranking_df["item_id"].isna().sum())
print("Score range:",
      round(popularity_ranking_df["popularity_score"].min(), 6),
      "to",
      round(popularity_ranking_df["popularity_score"].max(), 6))

Popularity ranking size: 309395
Any missing scores?: 0
Any missing item IDs?: 0
Score range: 1.872657 to 4.948716


In [60]:
# Optional: inspect the most popular items
# This helps verify that the ranking looks reasonable before generating recommendations

top_popular_items_df = popularity_ranking_df.head(20).copy()
display(top_popular_items_df)

,item_id,item_review_count,item_mean_rating,popularity_score,pop_rank
0,Hotel_Review-g274707-d658737-Reviews-Hotel_Res...,815,4.968098,4.948716,1
1,Hotel_Review-g188671-d1534996-Reviews-Huis_Kon...,215,5.000000,4.928243,2
2,Hotel_Review-g274887-d635259-Reviews-Kapital_I...,287,4.979094,4.925510,3
3,Hotel_Review-g188671-d590867-Reviews-Cote_Cana...,297,4.976431,4.924700,4
4,Hotel_Review-g298564-d1547516-Reviews-Hotel_Mu...,334,4.964072,4.918411,5
5,Hotel_Review-g186612-d213125-Reviews-Friars_Gl...,458,4.949782,4.916556,6
6,Hotel_Review-g293924-d7180030-Reviews-Hanoi_La...,974,4.929158,4.913589,7
7,Hotel_Review-g186370-d2001897-Reviews-Hawkins_...,196,4.989796,4.912526,8
8,Hotel_Review-g471846-d646675-Reviews-Dulini_Lo...,214,4.981308,4.910692,9
9,Hotel_Review-g188675-d2173238-Reviews-Main_Str...,211,4.981043,4.909522,10


In [61]:
# SECTION 3 — Build user history sets and recommendation helper functions

# For validation:
#   filter out items seen in TRAIN
#
# For test:
#   filter out items seen in TRAIN + VALID
#
# This matches your leave-last-out split design:
# - valid predicts the next item after train
# - test predicts the next item after train + valid

train_seen_by_user = train_df.groupby("user_id")["item_id"].agg(set).to_dict()

train_valid_df = pd.concat([train_df, valid_df], ignore_index=True)
train_valid_seen_by_user = train_valid_df.groupby("user_id")["item_id"].agg(set).to_dict()

print("Users with train history:", len(train_seen_by_user))
print("Users with train+valid history:", len(train_valid_seen_by_user))

Users with train history: 1927834
Users with train+valid history: 1927834


In [62]:
# Convert the global ranking to a Python list for fast recommendation generation

global_ranked_items = popularity_ranking_df["item_id"].tolist()

print("Number of ranked items:", len(global_ranked_items))
print("Top 10 globally ranked items:")
print(global_ranked_items[:10])

Number of ranked items: 309395
Top 10 globally ranked items:
['Hotel_Review-g274707-d658737-Reviews-Hotel_Residence_Agnes-Prague_Bohemia.html', 'Hotel_Review-g188671-d1534996-Reviews-Huis_Koning-Bruges_West_Flanders_Province.html', 'Hotel_Review-g274887-d635259-Reviews-Kapital_Inn-Budapest_Central_Hungary.html', 'Hotel_Review-g188671-d590867-Reviews-Cote_Canal-Bruges_West_Flanders_Province.html', 'Hotel_Review-g298564-d1547516-Reviews-Hotel_Mume-Kyoto_Kyoto_Prefecture_Kinki.html', 'Hotel_Review-g186612-d213125-Reviews-Friars_Glen-Killarney_County_Kerry.html', 'Hotel_Review-g293924-d7180030-Reviews-Hanoi_La_Siesta_Hotel_Spa-Hanoi.html', 'Hotel_Review-g186370-d2001897-Reviews-Hawkins_of_Bath-Bath_Somerset_England.html', 'Hotel_Review-g471846-d646675-Reviews-Dulini_Lodge-Sabi_Sand_Game_Reserve_Kruger_National_Park.html', 'Hotel_Review-g188675-d2173238-Reviews-Main_Street_Boutique_Hotel-Ieper_Ypres_West_Flanders_Province.html']


In [63]:
# Recommendation helper:
# returns the top-K globally popular items that the user has NOT already seen

def recommend_top_k_popular(user_id, seen_items_by_user, ranked_items, k=10):
    seen = seen_items_by_user.get(user_id, set())
    
    recs = []
    for item_id in ranked_items:
        if item_id not in seen:
            recs.append(item_id)
        if len(recs) >= k:
            break
    
    return recs

In [64]:
# Build ground-truth lookup tables
# Because your split is leave-last-out, each user should have exactly one target item in valid and one in test.

valid_target_by_user = dict(zip(valid_df["user_id"], valid_df["item_id"]))
test_target_by_user  = dict(zip(test_df["user_id"], test_df["item_id"]))

print("Validation users:", len(valid_target_by_user))
print("Test users:", len(test_target_by_user))

Validation users: 1927834
Test users: 1927834


In [65]:
# Preview recommendations for a few validation users

sample_valid_users = list(valid_target_by_user.keys())[:5]

preview_rows = []
for user_id in sample_valid_users:
    recs = recommend_top_k_popular(
        user_id=user_id,
        seen_items_by_user=train_seen_by_user,
        ranked_items=global_ranked_items,
        k=10
    )
    preview_rows.append({
        "user_id": user_id,
        "true_valid_item": valid_target_by_user[user_id],
        "top_10_recs": recs
    })

preview_valid_recs_df = pd.DataFrame(preview_rows)
display(preview_valid_recs_df)

,user_id,true_valid_item,top_10_recs
0,Navigate26266,Hotel_Review-g33857-d80762-Reviews-Days_Inn_by...,[Hotel_Review-g274707-d658737-Reviews-Hotel_Re...
1,LightPacker957,Hotel_Review-g188644-d206688-Reviews-Radisson_...,[Hotel_Review-g274707-d658737-Reviews-Hotel_Re...
2,GrandTour19809,Hotel_Review-g194863-d239260-Reviews-Villa_Ros...,[Hotel_Review-g274707-d658737-Reviews-Hotel_Re...
3,Storyteller17290,Hotel_Review-g187870-d235821-Reviews-Pensione_...,[Hotel_Review-g274707-d658737-Reviews-Hotel_Re...
4,GoPlaces25384,Hotel_Review-g150797-d226263-Reviews-Hotel_Jac...,[Hotel_Review-g274707-d658737-Reviews-Hotel_Re...


In [66]:
# Optional: verify none of the recommended items are already in the user's seen history

def count_seen_item_leaks(users, targets, seen_items_by_user, ranked_items, k=10):
    leaks = 0
    for user_id in users:
        recs = recommend_top_k_popular(user_id, seen_items_by_user, ranked_items, k=k)
        seen = seen_items_by_user.get(user_id, set())
        if any(item in seen for item in recs):
            leaks += 1
    return leaks

valid_leaks = count_seen_item_leaks(
    users=sample_valid_users,
    targets=valid_target_by_user,
    seen_items_by_user=train_seen_by_user,
    ranked_items=global_ranked_items,
    k=10
)

print("Recommendation leaks in preview users:", valid_leaks)

Recommendation leaks in preview users: 0


In [67]:
# SECTION 4 — Evaluate the popularity baseline on the VALIDATION split
#
# Metrics:
# - Hit@K: did the true item appear in the top-K recommendations?
# - NDCG@K: gives more credit when the true item appears higher in the ranking
#
# Because each user has exactly one held-out validation item, these are natural ranking metrics.

import math

In [85]:
# Metric helpers

import math

def hit_at_k(recommended_items, true_item):
    return 1.0 if true_item in recommended_items else 0.0

def ndcg_at_k(recommended_items, true_item):
    if true_item not in recommended_items:
        return 0.0
    rank_position = recommended_items.index(true_item) + 1
    return 1.0 / math.log2(rank_position + 1)

def precision_at_k(recommended_items, true_item, k):
    # With a single held-out item: 1/K if hit, 0 otherwise
    return (1.0 / k) if true_item in recommended_items else 0.0

In [86]:
# Evaluate for one K value (ranking metrics)

def evaluate_popularity_split(target_by_user, seen_items_by_user, ranked_items, k=10):
    rows = []
    
    for user_id, true_item in target_by_user.items():
        recs = recommend_top_k_popular(
            user_id=user_id,
            seen_items_by_user=seen_items_by_user,
            ranked_items=ranked_items,
            k=k
        )
        
        rows.append({
            "user_id": user_id,
            "true_item": true_item,
            "hit": hit_at_k(recs, true_item),
            "ndcg": ndcg_at_k(recs, true_item),
            "precision": precision_at_k(recs, true_item, k),
            "num_recs_returned": len(recs)
        })
    
    results_df = pd.DataFrame(rows)
    
    summary = {
        "k": k,
        "n_users_evaluated": len(results_df),
        "Hit@K": results_df["hit"].mean(),
        "NDCG@K": results_df["ndcg"].mean(),
        "Precision@K": results_df["precision"].mean(),
        "avg_num_recs_returned": results_df["num_recs_returned"].mean()
    }
    
    return results_df, pd.DataFrame([summary])

In [87]:
# Run validation evaluation at K=10

K = 10

valid_user_results_df, valid_summary_df = evaluate_popularity_split(
    target_by_user=valid_target_by_user,
    seen_items_by_user=train_seen_by_user,
    ranked_items=global_ranked_items,
    k=K
)

display(valid_summary_df)

,k,n_users_evaluated,Hit@K,NDCG@K,Precision@K,avg_num_recs_returned
0,10,1927834,0.000313,0.00017,0.000031,10.0


In [88]:
# Optional: also evaluate a few common K values for a broader view

k_values = [5, 10, 20]

valid_multi_k_summaries = []
for k in k_values:
    _, summary_df_k = evaluate_popularity_split(
        target_by_user=valid_target_by_user,
        seen_items_by_user=train_seen_by_user,
        ranked_items=global_ranked_items,
        k=k
    )
    valid_multi_k_summaries.append(summary_df_k)

valid_multi_k_summary_df = pd.concat(valid_multi_k_summaries, ignore_index=True)
display(valid_multi_k_summary_df)

,k,n_users_evaluated,Hit@K,NDCG@K,Precision@K,avg_num_recs_returned
0,5,1927834,0.000147,0.000115,0.000029,5.0
1,10,1927834,0.000313,0.000170,0.000031,10.0
2,20,1927834,0.000539,0.000226,0.000027,20.0


In [89]:
# RMSE evaluation — rating prediction perspective
# For the popularity baseline, the "predicted rating" for any item is its Bayesian smoothed score.
# For items unseen in training, we fall back to the global mean.
#
# This evaluates a fundamentally different question than ranking metrics:
# "How close is our predicted rating to the actual rating?"

import numpy as np

# Build a lookup from item_id to popularity score
item_score_lookup = dict(zip(
    train_item_stats["item_id"],
    train_item_stats["popularity_score"]
))

def compute_rmse(eval_df, item_score_lookup, global_mean):
    predicted = eval_df["item_id"].map(item_score_lookup).fillna(global_mean)
    actual = eval_df["rating"]
    rmse = np.sqrt(((predicted - actual) ** 2).mean())
    mae = (predicted - actual).abs().mean()
    n_unseen = eval_df["item_id"].map(item_score_lookup).isna().sum()
    return rmse, mae, n_unseen

valid_rmse, valid_mae, valid_unseen = compute_rmse(valid_df, item_score_lookup, global_mean_rating)
test_rmse, test_mae, test_unseen = compute_rmse(test_df, item_score_lookup, global_mean_rating)

rating_pred_df = pd.DataFrame([
    {"split": "valid", "RMSE": round(valid_rmse, 6), "MAE": round(valid_mae, 6),
     "n_evaluated": len(valid_df), "n_unseen_items_fallback": valid_unseen},
    {"split": "test", "RMSE": round(test_rmse, 6), "MAE": round(test_mae, 6),
     "n_evaluated": len(test_df), "n_unseen_items_fallback": test_unseen},
])

print("Rating prediction metrics (predicted = Bayesian avg, fallback = global mean):")
display(rating_pred_df)

Rating prediction metrics (predicted = Bayesian avg, fallback = global mean):


,split,RMSE,MAE,n_evaluated,n_unseen_items_fallback
0,valid,0.934757,0.725877,1927834,997
1,test,0.966656,0.750443,1927834,2458


In [90]:
# ══════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY — Popularity Baseline (All Metrics)
# ══════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("POPULARITY BASELINE — COMPLETE EVALUATION SUMMARY")
print("=" * 70)

print(f"\nModel: Bayesian-smoothed popularity ranking (non-personalized)")
print(f"Smoothing constant m = {m} (median train item review count)")
print(f"Global mean train rating C = {global_mean_rating:.4f}")
print(f"Items in ranking: {len(global_ranked_items):,}")

# ── Ranking Metrics ──────────────────────────────────────────────────────
print("\n" + "─" * 70)
print("RANKING METRICS (top-K recommendation)")
print("─" * 70)
print("Task: predict which item the user interacts with next")
print("Each user has exactly 1 held-out target (leave-last-out)")

ranking_summary = pd.concat(
    [
        valid_multi_k_summary_df.assign(split="valid"),
        test_multi_k_summary_df.assign(split="test")
    ],
    ignore_index=True
)[["split", "k", "n_users_evaluated", "Hit@K", "NDCG@K", "Precision@K"]].sort_values(
    ["k", "split"]
).reset_index(drop=True)

# Format for readability
ranking_display = ranking_summary.copy()
for col in ["Hit@K", "NDCG@K", "Precision@K"]:
    ranking_display[col] = ranking_display[col].map(lambda x: f"{x:.6f}")
ranking_display["n_users_evaluated"] = ranking_display["n_users_evaluated"].map(lambda x: f"{x:,}")

display(ranking_display)

# ── Rating Prediction Metrics ────────────────────────────────────────────
print("\n" + "─" * 70)
print("RATING PREDICTION METRICS")
print("─" * 70)
print("Task: predict the rating a user gives to an item")
print("Predicted rating = item Bayesian avg (fallback: global mean for unseen items)")

display(rating_pred_df)

# ── Context ──────────────────────────────────────────────────────────────
print("\n" + "─" * 70)
print("INTERPRETATION")
print("─" * 70)
print(f"""
This is a non-personalized baseline: every user receives the same ranked
list (minus items they've already seen). With ~{len(global_ranked_items):,} items and only
K=10 recommendations, the chance of hitting any specific user's held-out
item is inherently low (~{10/len(global_ranked_items):.5f} for random).

The baseline achieves Hit@10 roughly 10x better than random, confirming
that popular items do carry some signal, but personalization is needed
to meaningfully improve performance.

These results serve as the FLOOR that any personalized model (MF, etc.)
must beat to justify its complexity.
""")

POPULARITY BASELINE — COMPLETE EVALUATION SUMMARY

Model: Bayesian-smoothed popularity ranking (non-personalized)
Smoothing constant m = 18 (median train item review count)
Global mean train rating C = 4.0711
Items in ranking: 309,395

──────────────────────────────────────────────────────────────────────
RANKING METRICS (top-K recommendation)
──────────────────────────────────────────────────────────────────────
Task: predict which item the user interacts with next
Each user has exactly 1 held-out target (leave-last-out)


,split,k,n_users_evaluated,Hit@K,NDCG@K,Precision@K
0,test,5,"1,927,834",0.000179,0.000151,nan
1,valid,5,"1,927,834",0.000147,0.000115,0.000029
2,test,10,"1,927,834",0.000339,0.000204,nan
3,valid,10,"1,927,834",0.000313,0.000170,0.000031
4,test,20,"1,927,834",0.000570,0.000260,nan
5,valid,20,"1,927,834",0.000539,0.000226,0.000027



──────────────────────────────────────────────────────────────────────
RATING PREDICTION METRICS
──────────────────────────────────────────────────────────────────────
Task: predict the rating a user gives to an item
Predicted rating = item Bayesian avg (fallback: global mean for unseen items)


,split,RMSE,MAE,n_evaluated,n_unseen_items_fallback
0,valid,0.934757,0.725877,1927834,997
1,test,0.966656,0.750443,1927834,2458



──────────────────────────────────────────────────────────────────────
INTERPRETATION
──────────────────────────────────────────────────────────────────────

This is a non-personalized baseline: every user receives the same ranked
list (minus items they've already seen). With ~309,395 items and only
K=10 recommendations, the chance of hitting any specific user's held-out
item is inherently low (~0.00003 for random).

The baseline achieves Hit@10 roughly 10x better than random, confirming
that popular items do carry some signal, but personalization is needed
to meaningfully improve performance.

These results serve as the FLOOR that any personalized model (MF, etc.)
must beat to justify its complexity.



In [72]:
# SECTION 5 — Evaluate the popularity baseline on the TEST split
#
# For test:
# - recommendations should exclude items seen in TRAIN + VALID
# - the held-out target is the user's final test item

K = 10

test_user_results_df, test_summary_df = evaluate_popularity_split(
    target_by_user=test_target_by_user,
    seen_items_by_user=train_valid_seen_by_user,
    ranked_items=global_ranked_items,
    k=K
)

display(test_summary_df)

,k,n_users_evaluated,Hit@K,NDCG@K,avg_num_recs_returned
0,10,1927834,0.000339,0.000204,10.0


In [73]:
# Optional: evaluate a few common K values on TEST as well

k_values = [5, 10, 20]

test_multi_k_summaries = []
for k in k_values:
    _, summary_df_k = evaluate_popularity_split(
        target_by_user=test_target_by_user,
        seen_items_by_user=train_valid_seen_by_user,
        ranked_items=global_ranked_items,
        k=k
    )
    test_multi_k_summaries.append(summary_df_k)

test_multi_k_summary_df = pd.concat(test_multi_k_summaries, ignore_index=True)
display(test_multi_k_summary_df)

,k,n_users_evaluated,Hit@K,NDCG@K,avg_num_recs_returned
0,5,1927834,0.000179,0.000151,5.0
1,10,1927834,0.000339,0.000204,10.0
2,20,1927834,0.000570,0.000260,20.0


In [74]:
# Compare VALID vs TEST side by side

valid_compare_df = valid_multi_k_summary_df.copy()
valid_compare_df["split_name"] = "valid"

test_compare_df = test_multi_k_summary_df.copy()
test_compare_df["split_name"] = "test"

split_comparison_df = pd.concat([valid_compare_df, test_compare_df], ignore_index=True)
split_comparison_df = split_comparison_df[
    ["split_name", "k", "n_users_evaluated", "Hit@K", "NDCG@K", "avg_num_recs_returned"]
].sort_values(["k", "split_name"]).reset_index(drop=True)

display(split_comparison_df)

,split_name,k,n_users_evaluated,Hit@K,NDCG@K,avg_num_recs_returned
0,test,5,1927834,0.000179,0.000151,5.0
1,valid,5,1927834,0.000147,0.000115,5.0
2,test,10,1927834,0.000339,0.000204,10.0
3,valid,10,1927834,0.000313,0.000170,10.0
4,test,20,1927834,0.000570,0.000260,20.0
5,valid,20,1927834,0.000539,0.000226,20.0


In [75]:
# SECTION 6 — Final baseline summary and reusable inference helper
#
# This section:
# 1) creates one compact summary table for the popularity baseline
# 2) saves a small config/metadata file
# 3) provides a simple function you can reuse later to get recommendations for any user

import json

In [76]:
# Combine the key results into one compact summary table

final_baseline_summary_df = pd.concat(
    [
        valid_multi_k_summary_df.assign(split_name="valid"),
        test_multi_k_summary_df.assign(split_name="test")
    ],
    ignore_index=True
)

final_baseline_summary_df = final_baseline_summary_df[
    ["split_name", "k", "n_users_evaluated", "Hit@K", "NDCG@K", "avg_num_recs_returned"]
].sort_values(["split_name", "k"]).reset_index(drop=True)

display(final_baseline_summary_df)

,split_name,k,n_users_evaluated,Hit@K,NDCG@K,avg_num_recs_returned
0,test,5,1927834,0.000179,0.000151,5.0
1,test,10,1927834,0.000339,0.000204,10.0
2,test,20,1927834,0.000570,0.000260,20.0
3,valid,5,1927834,0.000147,0.000115,5.0
4,valid,10,1927834,0.000313,0.000170,10.0
5,valid,20,1927834,0.000539,0.000226,20.0


In [77]:
# Save a small baseline config so the setup is easy to document later
import json
popularity_baseline_config = {
    "model_name": "smoothed_popularity_baseline",
    "train_file": os.path.basename(TRAIN_PATH),
    "valid_file": os.path.basename(VALID_PATH),
    "test_file": os.path.basename(TEST_PATH),
    "score_definition": "bayesian_smoothed_mean_rating",
    "score_formula": "score_i = (v/(v+m))*R + (m/(v+m))*C",
    "v_definition": "item_review_count_in_train",
    "R_definition": "item_mean_rating_in_train",
    "C_definition": "global_mean_rating_in_train",
    "m_definition": "median_item_review_count_in_train",
    "m_value": int(m),
    "global_mean_rating": float(global_mean_rating),
    "ranking_type": "global_non_personalized_item_ranking",
    "valid_seen_filter": "exclude items seen in train",
    "test_seen_filter": "exclude items seen in train+valid",
    "metrics": ["Hit@K", "NDCG@K"],
    "k_values_evaluated": [5, 10, 20]
}

config_path = os.path.join(EXPORT_DIR, "popularity_baseline_config.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(popularity_baseline_config, f, indent=2)

print("Saved config to:", config_path)

Saved config to: D:\256_group_dataset\phase1_outputs\popularity_baseline_config.json


In [78]:
# Reusable helper for getting recommendations for a single user
#
# split_name='valid'  -> excludes TRAIN history
# split_name='test'   -> excludes TRAIN + VALID history

def get_popularity_recommendations(user_id, split_name="test", k=10):
    if split_name == "valid":
        seen_items = train_seen_by_user
    elif split_name == "test":
        seen_items = train_valid_seen_by_user
    else:
        raise ValueError("split_name must be 'valid' or 'test'")
    
    recs = recommend_top_k_popular(
        user_id=user_id,
        seen_items_by_user=seen_items,
        ranked_items=global_ranked_items,
        k=k
    )
    
    rec_df = popularity_ranking_df[
        popularity_ranking_df["item_id"].isin(recs)
    ][["item_id", "pop_rank", "item_review_count", "item_mean_rating", "popularity_score"]].copy()
    
    rec_df["recommendation_order"] = range(1, len(rec_df) + 1)
    rec_df = rec_df.sort_values("recommendation_order").reset_index(drop=True)
    
    return rec_df

In [79]:
# Example usage on one validation user and one test user

example_valid_user = next(iter(valid_target_by_user.keys()))
example_test_user = next(iter(test_target_by_user.keys()))

print("Example VALID user:", example_valid_user)
display(get_popularity_recommendations(example_valid_user, split_name="valid", k=10))

print("Example TEST user:", example_test_user)
display(get_popularity_recommendations(example_test_user, split_name="test", k=10))

Example VALID user: Navigate26266


,item_id,pop_rank,item_review_count,item_mean_rating,popularity_score,recommendation_order
0,Hotel_Review-g274707-d658737-Reviews-Hotel_Res...,1,815,4.968098,4.948716,1
1,Hotel_Review-g188671-d1534996-Reviews-Huis_Kon...,2,215,5.000000,4.928243,2
2,Hotel_Review-g274887-d635259-Reviews-Kapital_I...,3,287,4.979094,4.925510,3
3,Hotel_Review-g188671-d590867-Reviews-Cote_Cana...,4,297,4.976431,4.924700,4
4,Hotel_Review-g298564-d1547516-Reviews-Hotel_Mu...,5,334,4.964072,4.918411,5
5,Hotel_Review-g186612-d213125-Reviews-Friars_Gl...,6,458,4.949782,4.916556,6
6,Hotel_Review-g293924-d7180030-Reviews-Hanoi_La...,7,974,4.929158,4.913589,7
7,Hotel_Review-g186370-d2001897-Reviews-Hawkins_...,8,196,4.989796,4.912526,8
8,Hotel_Review-g471846-d646675-Reviews-Dulini_Lo...,9,214,4.981308,4.910692,9
9,Hotel_Review-g188675-d2173238-Reviews-Main_Str...,10,211,4.981043,4.909522,10


Example TEST user: Navigate26266


,item_id,pop_rank,item_review_count,item_mean_rating,popularity_score,recommendation_order
0,Hotel_Review-g274707-d658737-Reviews-Hotel_Res...,1,815,4.968098,4.948716,1
1,Hotel_Review-g188671-d1534996-Reviews-Huis_Kon...,2,215,5.000000,4.928243,2
2,Hotel_Review-g274887-d635259-Reviews-Kapital_I...,3,287,4.979094,4.925510,3
3,Hotel_Review-g188671-d590867-Reviews-Cote_Cana...,4,297,4.976431,4.924700,4
4,Hotel_Review-g298564-d1547516-Reviews-Hotel_Mu...,5,334,4.964072,4.918411,5
5,Hotel_Review-g186612-d213125-Reviews-Friars_Gl...,6,458,4.949782,4.916556,6
6,Hotel_Review-g293924-d7180030-Reviews-Hanoi_La...,7,974,4.929158,4.913589,7
7,Hotel_Review-g186370-d2001897-Reviews-Hawkins_...,8,196,4.989796,4.912526,8
8,Hotel_Review-g471846-d646675-Reviews-Dulini_Lo...,9,214,4.981308,4.910692,9
9,Hotel_Review-g188675-d2173238-Reviews-Main_Str...,10,211,4.981043,4.909522,10
